# B02 · Numerical embeddings and ensembles

PROVIDED: authenticated full selected-prediction evidence and visible scorer. TODO: three live functions. CHECK: behavioral cases, leakage example and independent score replay. EXIT: written defense. This portable lane replays fresh author evidence; it does not launch new training or paid work. Python 3.12 + NumPy; install NumPy if absent before starting.

In [1]:
import base64,copy,hashlib,io,json,math,statistics,tempfile,zipfile
from pathlib import Path
import numpy as np
workspace=Path(tempfile.mkdtemp(prefix="b02-notebook-"))

# Numerical embeddings and ensembles

<p class="eyebrow">B02 · Year 5 → 6 bridge · 25-minute core + reproduction lab</p>

**Your win:** distinguish a better input representation from a better ensemble, then design a comparison that could tell them apart.

[B01](https://avistian.github.io/relational/lessons/b01-architecture-coverage-honest-comparison.html) showed why a score compares an entire pipeline. Its unresolved question: once the input table is matched, which part of the predictor deserves credit? Here we separate two choices: how a scalar enters a neural network, and how several predictions are trained and combined. A strong relational result must eventually beat credible flat-table baselines, not an unnecessarily weak MLP.

Prerequisites: [RealMLP, L053](https://avistian.github.io/relational/lessons/0053-realmlp-strong-defaults.html), [TabM, L054](https://avistian.github.io/relational/lessons/0054-tabm-parameter-efficient-ensembling.html), and an affine layer `x @ W + b`. You need only shapes, averages and squared error for the core lesson.

## 1. Give a scalar useful coordinates

An ordinary MLP receives a numerical feature as one number. An embedding maps that number into several coordinates **before mixing different features**. The extra coordinates can make useful nonlinear patterns easier for subsequent layers to express; they do not create new observations or guarantee better generalization. [Numerical embeddings paper, §§3–4](https://arxiv.org/abs/2203.05556).

For one feature with training-fitted knots `[0, 2, 4]`, piecewise-linear encoding gives `x=1 → [0.5, 0]` and `x=3 → [1, 0.5]`. The first coordinate fills the first interval, then the second fills the next. This version extrapolates at the ends: `−1 → [−0.5, 0]`, `5 → [1, 1.5]`. Fit knots on **training features only**. Distinct knots are required; ties need an explicit preprocessing policy. Our exercise rejects duplicate edges instead of silently inventing widths.

Periodic embeddings instead expose learned oscillations. The pinned TabPack source computes `α ⊙ concat(x, cos(2π(wx+b))) + β` per feature. It starts the affine scale and shift at zero, so the initial embedding is zero; the affine parameters can receive gradients first. The implementation includes `2π` and truncated-normal initialization; the paper's compact equation omits that explicit factor. We preserve the code formula for reproduction. These embeddings are not interchangeable with the earlier paper's piecewise-linear or PLR recipes. [TabPack Appendix B](https://arxiv.org/html/2607.05380v1#A2) · [Pinned implementation](https://github.com/yandex-research/tabpack/blob/05a89e21b955f12de84889d662e15ca534019aaa/src/project/nn.py).

End-to-end architecture

## 2. Follow the weights, not the word “ensemble”

**TabM:** several prediction paths share large weight matrices. For one affine layer in row-vector notation, member `k` computes `((x ⊙ r_k) W) ⊙ s_k + b_k`. Its elementwise adapters differ, while `W` is shared. Train with the **mean of member losses**, not the loss of the averaged prediction. At inference, average regression predictions or classification probabilities. The full architecture includes nonlinearities, initialization rules and separate output heads. The layer exercise is deliberately only one load-bearing operator. [TabM §3](https://arxiv.org/html/2410.24210v1#S3).

A tiny trace: `x=[1,2]`, `W=[[2,−1],[0.5,3]]`. With adapters all ones and zero bias, the output is `[3,5]`. With `r=[−1,2]`, `s=[2,−1]`, it is `[0,−13]`. The same matrix can support different member predictions. A rank-one adapter mask does not make the effective matrix rank one.

**TabPack:** pack many independently parameterized MLPs into batched operations. Members may differ in depth, dropout, embedding size and optimizer settings; padding and masks make heterogeneous shapes computable together. Each member owns its weights. Its packed optimizer preserves member-specific state and hyperparameters. The online ensemble is selected using validation predictions as training proceeds. Efficiency comes from executing the pack together, not from sharing TabM's backbone matrix. [TabPack §3, Figure 2](https://arxiv.org/html/2607.05380v1#S3).

The official operator audit checks heterogeneous packed affine outputs and gradients against explicit loops, zero cross-member weight gradients, and cosine embedding arithmetic. It validates those operators, not full training-trajectory identity. Read the [audit receipt](https://avistian.github.io/relational/labs/evidence/b02/mechanism-audit.json).

## 3. Selection is work, even inside one run

Suppose validation labels are `[1,1]`. Member A predicts `[0,2]`, B predicts `[2,0]`, C predicts `[5,5]`. A and B each have MSE 1; their average has MSE 0. A greedy selector chooses A on the first tie, adds B, then stops because no candidate improves the average.

Now use the exercise to change the selection labels. The deliberately contaminated option makes C win. It demonstrates how test-guided selection changes the deployed predictor; it is not an authorized evaluation recipe.

**Try it in code:** call your selector with [1,1] and then [5,5]. Explain why only the first is a valid selection procedure.

Our fixed-pool exercise allows repeated indices, chooses the first exact tie and stops on strict non-improvement. TabPack's full online procedure additionally tracks training states, member stopping, ensemble history and reuse of the current ensemble. The upstream implementation, not this small exercise, runs the reproduction. Internal validation choices count as selection even when there is no outer hyperparameter search. [TabPack §§3.5–3.6 and Appendix C](https://arxiv.org/html/2607.05380v1).

**Predict before revealing:** Does averaging guarantee a better predictor than a separately trained MLP?

<details><summary>Check your reasoning</summary>No. Convexity says the loss of an average is no worse than the average loss of those same members for squared error. It does not compare against a separately optimized baseline, guarantee better classification error, or remove selection bias.</details>

## 4. Build the experiment that answers your question

A four-cell design isolates embedding on/off and ensembling on/off. Hold dataset, preprocessing, optimizer family, training/stopping policy and candidate budget fixed. For a TabM-family experiment, define the single-path control and verify its initialization; merely replacing a name does not guarantee a matched control.

| Cell | Embedding | Member paths | What the paired contrast asks |
|---|---|---|---|
| A | Off | One | Reference MLP |
| B | On | One | B − A: embedding effect for one path |
| C | Off | Many | C − A: ensemble effect without embedding |
| D | On | Many | D − C: embedding effect with ensemble |

For RMSE, improvement is a negative difference. The interaction is `(D − C) − (B − A)`. Report paired seed contrasts on the same rows. Match **selection opportunity** and separately report actual fit/inference cost; equal candidate counts do not make resource consumption equal. State whether widths, parameters or compute are controlled, because adding embeddings changes input dimension. Do not assert all three are simultaneously matched.

Add a tuned tree and a RealMLP baseline on the same split, with declared preprocessing and equal outer-search allowances. Keep their purpose clear: they test whether the resulting neural pipeline is useful, while the four cells test the mechanism. TabPack with and without embeddings is not a clean embedding-only ablation: the published depth ranges also differ. The selected experiment below does **not** execute this factorial design or those baselines. [TabPack Appendix F](https://arxiv.org/html/2607.05380v1#A6).

**Your task:** specify six candidate recipes per arm, three paired seeds, a validation-only stopping rule and a maximum total cost. Count all fits, internal selection decisions, failed runs and final evaluations. This is a planning exercise, not permission to launch it. Give two falsification tests: one that would expose leakage and one that would disprove your claimed source of improvement.

## 5. Read the reproduction honestly

**COMPLETE_SELECTED_RELEASE_PROTOCOL** · five evaluation seeds after a fresh 64-member search.

| Evaluation seed | Independently scored test RMSE |
|---|---:|
| 0 | 0.41779797 |
| 1 | 0.41679763 |
| 2 | 0.41797847 |
| 3 | 0.41725860 |
| 4 | 0.41632376 |

Mean **0.41723129**, sample SD **0.00068772**. Released five-seed mean 0.41757790. Independent scoring reconciles all 44,586 validation/test predictions (search plus five evaluations), with maximum report difference 1.9e-08. Seeds share one split, so this is conditional training variability, not cross-dataset uncertainty.

**Evidence-saving deviation:** upstream writes an ensemble name instead of prediction arrays. The first unmodified run is retained. A complete rerun uses a read-only observer of the original final in-memory member predictions; model/selection code stays unchanged. All reruns count toward the budget.


The named lane is **B02-TABPACK-CALIFORNIA**, the complete selected **release protocol**. It includes the original 64-member search, its validation-selected member configurations, and all five evaluation seeds. All 13,209 training, 3,303 validation and 4,128 test rows are retained; source and array hashes pin positional row identity. The source controls noisy-quantile preprocessing, target normalization, Muon/AdamW, bfloat16, member patience 16 and ensemble patience 32. The fresh search determines its own selected configurations; copying the released 12-member list would omit selection cost.

The paper's Table 14 caption says ten seeds, but the pinned generator, evaluation config and report use five (`0…4`). We retain that mismatch. Fresh release-protocol completion is not whole-paper reproduction. The full benchmark, original numerical-embedding benchmark, fresh TabM benchmark and tuned tree/RealMLP comparisons are **NOT_RUN**. A close scalar score cannot establish historical environment identity or the causal effect of embeddings.

[Full protocol and commands](https://avistian.github.io/relational/labs/b02-reproduction.md) · [Evidence report](https://avistian.github.io/relational/labs/evidence/b02/report.json) · [Student notebook](https://avistian.github.io/relational/labs/b02-numerical-embeddings-and-ensembles.ipynb) · [Executed solution](https://avistian.github.io/relational/labs/solutions/b02-numerical-embeddings-and-ensembles.ipynb) · [Notebook HTML](https://avistian.github.io/relational/labs/html/b02-numerical-embeddings-and-ensembles.html).

## 6. Earn the distinction

In the notebook, implement three live functions: piecewise-linear encoding, member-wise affine computation and validation-only greedy selection. Your functions feed behavioral checks and the worked selection result. Then independently replay every retained fresh prediction. Reading a solution or passing author checks does not complete your defense.

**Exit:** explain in 200–300 words why TabM sharing, TabPack packing and numerical embeddings are three separate choices. Include the source-code versus paper discrepancies, one fair factorial comparison, baseline rationale, two falsification tests and one finding that would change your mind. Status stays **PENDING_WRITTEN_DEFENSE** until reviewed.

Revisit after 1, 7 and 30 days: draw the weight-sharing distinction from memory; derive the four-cell interaction; audit a new comparison. Next, [B03 in the bridge plan](https://avistian.github.io/relational/plan/year-5-6-bridge.md#b03) asks how a pretrained predictor changes the adaptation budget.

**Primary reading:** TabPack Figure 2 and §§3.3–3.6, then compare TabM §3 and the numerical embeddings paper §3. Use the [field guide](https://avistian.github.io/relational/reference/b02-embeddings-ensembles.html) for retrieval. Ask the agent about any unclear shape, gradient, selection rule or evidence boundary.


In [2]:
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='e665fb4b492348308ca7ec654a5e0ca52e2287051876f2ff18a356c97b3a46d0'
with zipfile.ZipFile(io.BytesIO(payload)) as z:z.extractall(workspace)

## SOLUTION · piecewise_linear

Return one coordinate per interval, extrapolate end bins, reject invalid knots. Train-fitted edges are provided as an argument.

In [3]:
def piecewise_linear(x, edges):
    """One feature, train-fitted strictly increasing knots; extrapolate end bins."""
    x=np.asarray(x,dtype=float);edges=np.asarray(edges,dtype=float)
    if edges.ndim!=1 or len(edges)<3 or not np.isfinite(edges).all() or np.any(np.diff(edges)<=0):
        raise ValueError('At least three finite, strictly increasing edges required')
    if not np.isfinite(x).all():raise ValueError('Finite inputs required')
    z=(x[...,None]-edges[:-1])/np.diff(edges)
    result=np.clip(z,0,1)
    result[...,0]=np.minimum(z[...,0],1)
    result[...,-1]=np.maximum(z[...,-1],0)
    return result

## SOLUTION · member_predictions

Return [batch,member,out] using shared W and member r/s/bias. Keep axes explicit; this is one affine layer.

In [4]:
def member_predictions(x, shared, r, s, bias):
    """One TabM-style affine layer: [batch,member,out]; no activation here."""
    x=np.asarray(x);shared=np.asarray(shared);r=np.asarray(r);s=np.asarray(s);bias=np.asarray(bias)
    if x.ndim!=2 or shared.ndim!=2 or r.ndim!=2 or s.ndim!=2 or bias.shape!=s.shape or r.shape[0]!=s.shape[0] or x.shape[1]!=r.shape[1] or shared.shape!=(r.shape[1],s.shape[1]):
        raise ValueError('Incompatible batch/member/feature dimensions')
    return np.einsum('bki,io->bko',x[:,None,:]*r[None,:,:],shared)*s[None,:,:]+bias[None,:,:]

## SOLUTION · greedy_validation

Use member-by-row validation predictions only. Permit repeated members; choose first ties; stop when MSE does not strictly improve. Do not import a finished selector.

In [5]:
def greedy_validation(predictions, target, max_size=32):
    """Fixed-pool regression exercise; allow repeats, first tie, strict improvement.

    This is not TabPack's online lifecycle or early-stopping implementation.
    Only validation arrays belong here; use selected indices unchanged on test.
    """
    p=np.asarray(predictions,dtype=float);y=np.asarray(target,dtype=float)
    if p.ndim!=2 or y.shape!=(p.shape[1],) or len(p)==0 or not np.isfinite(p).all() or not np.isfinite(y).all() or type(max_size)!=int or max_size<1:
        raise ValueError('Finite member-by-row predictions and positive size required')
    chosen=[];total=np.zeros_like(y);best=float('inf')
    for size in range(max_size):
        candidate=(total[None,:]+p)/(size+1)
        losses=np.mean((candidate-y)**2,axis=1)
        index=int(np.argmin(losses));loss=float(losses[index])
        if loss>=best:break
        chosen.append(index);total+=p[index];best=loss
    return chosen

## CHECK · feedback on your actual functions

In [6]:
def check_functions(ple, members, select):
    np.testing.assert_allclose(ple(np.array([-1.,0.,1.,2.,4.,5.]),np.array([0.,2.,4.])),[[-.5,0],[0,0],[.5,0],[1,0],[1,1],[1,1.5]])
    try:ple(np.array([0.]),np.array([0.,0.,1.]))
    except ValueError:pass
    else:raise AssertionError('Duplicate edges accepted')
    x=np.array([[1.,2.],[3.,4.]]);w=np.array([[2.,-1.],[.5,3.]])
    r=np.array([[1.,1.],[-1.,2.]]);s=np.array([[1.,1.],[2.,-1.]]);b=np.array([[.2,.3],[.4,.5]])
    oracle=np.array([[(x[i]*r[k])@w*s[k]+b[k] for k in range(2)] for i in range(2)])
    np.testing.assert_allclose(members(x,w,r,s,b),oracle)
    # Average of two individually imperfect members is exact; selecting on test would differ.
    p=np.array([[0.,2.],[2.,0.],[5.,5.]])
    ids=select(p,np.array([1.,1.]),4)
    assert ids==[0,1],ids
    assert select(np.array([[1.,1.],[1.,1.]]),np.array([1.,1.]),4)==[0]
    try:select(np.array([[np.nan]]),np.array([1.]))
    except ValueError:pass
    else:raise AssertionError('Nonfinite predictions accepted')
    return {'piecewise_extrapolation':'PASS','member_axes':'PASS','validation_selection_ties_stop':'PASS'}

In [7]:
print(check_functions(piecewise_linear,member_predictions,greedy_validation))
pool=np.array([[0.,2.],[2.,0.],[5.,5.]])
assert greedy_validation(pool,np.array([1.,1.]))==[0,1]
assert greedy_validation(pool,np.array([5.,5.]))==[2]
print('Test-guided selection changes the predictor: reject that workflow.')

{'piecewise_extrapolation': 'PASS', 'member_axes': 'PASS', 'validation_selection_ties_stop': 'PASS'}
Test-guided selection changes the predictor: reject that workflow.


## PROVIDED · independent full prediction replay

The following visible function authenticates every compact input and selected-member prediction; checks the complete row partition, search-to-evaluation configurations, seed coverage and member/checkpoint identities; and recomputes every validation/test RMSE with a scalar summation oracle.

In [8]:
def audit(directory):
    root=Path(directory)
    manifest=json.loads((root/'compact-lock.json').read_text())
    for name,digest in manifest.items():
        if hashlib.sha256((root/name).read_bytes()).hexdigest()!=digest:raise ValueError('Evidence hash mismatch: '+name)
    y=np.load(root/'data/y.npy');x=np.load(root/'data/x_num.npy')
    split={part:np.load(root/f'data/{part}.npy') for part in ['train','val','test']}
    assert {k:len(v) for k,v in split.items()}=={'train':13209,'val':3303,'test':4128}
    joined=np.concatenate(list(split.values()))
    assert len(joined)==len(y)==len(x)==20640 and len(np.unique(joined))==len(y)
    assert np.array_equal(np.sort(joined),np.arange(len(y)))
    main=json.loads((root/'runs/main/report.json').read_text())
    search=json.loads((root/'runs/main/config.json').read_text())
    assert search['seed']==0 and search['n_models']==64 and search['n_epochs']==-1 and search['patience']==16
    assert search['amp_dtype']=='bfloat16' and search['batch_size']==256
    configs=json.loads((root/'runs/main/experiments.json').read_text())
    selected_ids=sorted(set(main['online_ensembles']['greedy']['report']['ids']))
    evaluation=json.loads((root/'runs/evaluation/config.json').read_text())
    assert evaluation['n_seeds']==5
    assert evaluation['base_config']['configs']==[configs[i]['config'] for i in selected_ids]
    assert evaluation['base_config']['n_models']==len(selected_ids)
    report=json.loads((root/'runs/evaluation/report.json').read_text())
    assert [a['config']['seed'] for a in report['experiments']]==list(range(5))
    records=[];max_error=0.;member_predictions=0
    for name,r in [('main',main)]+[(str(i),e['report']) for i,e in enumerate(report['experiments'])]:
        er=r['online_ensembles']['greedy']['report']
        with np.load(root/f'runs/{name}/observed_ensemble.npz',allow_pickle=False) as z:
            assert z['ids'].tolist()==er['ids'] and z['steps'].tolist()==er['steps']
            scores={}
            for part in ['val','test']:
                p=z[part];assert p.shape==(len(er['ids']),len(split[part])) and np.isfinite(p).all()
                # Preserve float32 averaging in released prediction path; score independently in float64.
                prediction=p.mean(axis=0).astype(np.float64);target=y[split[part]].astype(np.float64)
                rmse=math.sqrt(math.fsum(float(a-b)**2 for a,b in zip(prediction,target))/len(target))
                oracle=float(np.sqrt(np.mean((prediction-target)**2)))
                assert abs(oracle-rmse)<1e-12
                error=abs(rmse-er['metrics'][part]['rmse']);assert error<1e-6,(name,part,error)
                max_error=max(max_error,error);scores[part]=rmse;member_predictions+=p.size
            records.append(dict(run=name,selected_member_occurrences=len(er['ids']),unique_members=len(set(er['ids'])),**scores))
    observed=[r['test'] for r in records[1:]]
    original=json.loads((root/'released-report.json').read_text())
    released=[e['report']['online_ensembles']['greedy']['report']['metrics']['test']['rmse'] for e in original['experiments']]
    assert len(released)==5
    return dict(status='COMPLETE_SELECTED_RELEASE_PROTOCOL',paper_parity='UNRESOLVED_SEED_COUNT',source_commit='05a89e21b955f12de84889d662e15ca534019aaa',search_members=64,selected_configurations=len(selected_ids),evaluation_seeds=list(range(5)),rows={k:len(v) for k,v in split.items()},runs=records,test_mean=statistics.mean(observed),test_sample_sd=statistics.stdev(observed),released_mean=statistics.mean(released),released_sample_sd=statistics.stdev(released),max_score_error=max_error,max_released_seed_score_difference=max(abs(a-b) for a,b in zip(observed,released)),scored_prediction_rows=6*(3303+4128),retained_member_prediction_values=member_predictions,source_serialization_bug='Original online_ensemble_predictions.npz stores a name; read-only observer saves original in-memory arrays',full_benchmarks='NOT_RUN',learner='PENDING_WRITTEN_DEFENSE')

In [9]:
report=audit(workspace/'compact')
assert report==json.loads((workspace/'report.json').read_text())
Path('b02-report.json').write_text(json.dumps(report,indent=2)+'\n')
print(json.dumps(report,indent=2))

{
  "status": "COMPLETE_SELECTED_RELEASE_PROTOCOL",
  "paper_parity": "UNRESOLVED_SEED_COUNT",
  "source_commit": "05a89e21b955f12de84889d662e15ca534019aaa",
  "search_members": 64,
  "selected_configurations": 13,
  "evaluation_seeds": [
    0,
    1,
    2,
    3,
    4
  ],
  "rows": {
    "train": 13209,
    "val": 3303,
    "test": 4128
  },
  "runs": [
    {
      "run": "main",
      "selected_member_occurrences": 20,
      "unique_members": 13,
      "val": 0.42022471744237516,
      "test": 0.4156024857972293
    },
    {
      "run": "0",
      "selected_member_occurrences": 12,
      "unique_members": 8,
      "val": 0.42568836698122003,
      "test": 0.4177979721694119
    },
    {
      "run": "1",
      "selected_member_occurrences": 21,
      "unique_members": 10,
      "val": 0.42214792276412455,
      "test": 0.41679763145163484
    },
    {
      "run": "2",
      "selected_member_occurrences": 20,
      "unique_members": 10,
      "val": 0.42124019934724966,
      "t

## Read the training implementation

Fresh training uses the original pinned release, not these NumPy exercises. [Pinned model/training code](https://github.com/yandex-research/tabpack/blob/05a89e21b955f12de84889d662e15ca534019aaa/src/project/tabpack.py), [packed layers](https://github.com/yandex-research/tabpack/blob/05a89e21b955f12de84889d662e15ca534019aaa/src/project/nn.py), [optimizers](https://github.com/yandex-research/tabpack/blob/05a89e21b955f12de84889d662e15ca534019aaa/src/project/optim.py). The local source archive accompanies the lesson; use the reproduction protocol to rebuild the full environment. The selected code below makes the actual embedding and weight ownership visible.

### Official LinearPack

```python
class LinearPack(ModulePack):
    """A pack of linear layers.

    Important implementation details:

    * The module assumes that the input is already properly masked according to
      `in_features` passed to the constructor. Therefore, `in_features` is only used in
      `reset_parameters`, but not in `forward`.
    * The module properly masks its output in `forward` according to `out_features`
      passed to the constructor.
    """

    def __init__(
        self,
        in_features: int | list[int],
        out_features: int | list[int],
        bias: bool = True,
        *,
        max_in_features: None | int = None,
        max_out_features: None | int = None,
        pack_size: int,
        loop: bool = False,
        dtype: None | torch.dtype = None,
        device: None | str | torch.device = None,
    ) -> None:
        assert pack_size > 0

        factory_kwargs: KWArgs = {'dtype': dtype, 'device': device}
        in_features_buffer, max_in_features = _prepare_dimension_pack(
            in_features, max_in_features, pack_size, **factory_kwargs
        )
        out_features_buffer, max_out_features = _prepare_dimension_pack(
            out_features, max_out_features, pack_size, **factory_kwargs
        )

        super().__init__()
        self.in_features = in_features_buffer
        self.out_features = out_features_buffer
        self.weight = ParameterPack(
            torch.empty(pack_size, max_out_features, max_in_features, **factory_kwargs)
        )
        self.bias = (
            ParameterPack(torch.empty(pack_size, max_out_features, **factory_kwargs))
            if bias
            else None
        )
        self._loop = loop

        self.reset_parameters()

    @property
    def max_in_features(self) -> int:
        return self.weight.shape[-1]

    @property
    def max_out_features(self) -> int:
        return self.weight.shape[-2]

    @property
    def pack_size(self) -> int:
        return get_pack_size(self.weight)

    def reset_parameters(self) -> None:
        if self.in_features is None:
            d_in_rsqrt = self.max_in_features**-0.5
            for p in (self.weight, self.bias):
                if p is not None:
                    nn.init.uniform_(p, -d_in_rsqrt, d_in_rsqrt)

        else:
            d_in_rsqrt = self.in_features.float().rsqrt()
            for p in (self.weight, self.bias):
                if p is not None:
                    single_shape = p.shape[1:]
                    pack_view_idx = (slice(None), *((None,) * (p.ndim - 1)))
                    p_init = torch.lerp(
                        -d_in_rsqrt[*pack_view_idx].expand(-1, *single_shape),
                        d_in_rsqrt[*pack_view_idx].expand(-1, *single_shape),
                        torch.rand(
                            self.pack_size,
                            *single_shape,
                            dtype=p.dtype,
                            device=p.device,
                        ),
                    )
                    p.data.copy_(p_init)

    def _do_forward(
        self,
        x: Tensor,
        *,
        weight: Tensor,
        bias: None | Tensor,
        out_features: None | Tensor,
    ) -> Tensor:
        x = (
            torch.bmm(x, weight.transpose(-2, -1))
            if bias is None
            else torch.baddbmm(bias.unsqueeze(BATCH_DIM), x, weight.transpose(-2, -1))
        )
        if out_features is not None:
            output_mask = (
                torch.arange(self.max_out_features, device=x.device)[None]
                < out_features[:, None]
            )
            x = x * output_mask.float().unsqueeze(BATCH_DIM)
        return x

    def forward(self, x: Tensor, pack_idx: None | Tensor = None) -> Tensor:
        assert x.ndim == 3  # (pack_size, batch_size, in_features)
        assert x.shape[-1] == self.max_in_features

        weight = _get_tensor_pack(self.weight, pack_idx)
        bias = _get_tensor_pack(self.bias, pack_idx)
        out_features = _get_tensor_pack(self.out_features, pack_idx)

        if self._loop:
            # This implementation is very slow and is needed only for debugging;
            # in particular, for achieving reproducibility between running `k`
            # models with `LinearPack(..., pack_size=1)` and running one model
            # with `LinearPack(..., pack_size=k)`.
            return torch.cat(
                [
                    self._do_forward(
                        x[i : i + 1],
                        weight=weight[i : i + 1],
                        bias=(None if bias is None else bias[i : i + 1]),
                        out_features=(
                            None if out_features is None else out_features[i : i + 1]
                        ),
                    )
                    for i in range(get_pack_size(x))
                ]
            )
        else:
            return self._do_forward(
                x, weight=weight, bias=bias, out_features=out_features
            )
```

### Official CosineEmbeddings

```python
class CosineEmbeddings(nn.Module):
    """Cosine feature embeddings."""

    def __init__(self, n_features: int, d_embedding: int, *, init_scale: float) -> None:
        # The original feature is concatenated to the trainable part of its embedding.
        # This is reflected in the following constant for better clarity of some
        # implementation details of this class.
        concat_input = True

        super().__init__()
        self.weight = nn.Parameter(
            torch.empty(n_features, d_embedding - int(concat_input))
        )
        self.bias = nn.Parameter(torch.empty(self.weight.shape))
        self.elementwise_affine_weight = nn.Parameter(
            torch.empty(
                *self.weight.shape[:-1], self.weight.shape[-1] + int(concat_input)
            )
        )
        self.elementwise_affine_bias = nn.Parameter(
            torch.empty(self.elementwise_affine_weight.shape)
        )

        self.d_embedding = d_embedding
        self.init_scale = init_scale
        self._concat_input = concat_input

        self.reset_parameters()

    def get_output_shape(self) -> torch.Size:
        return torch.Size(
            (self.weight.shape[0], self.weight.shape[1] + int(self._concat_input))
        )

    def reset_parameters(self) -> None:
        bound = 3.0
        nn.init.trunc_normal_(self.weight, a=-bound, b=bound)
        with torch.inference_mode():
            self.weight *= self.init_scale
        nn.init.zeros_(self.bias)

        nn.init.zeros_(self.elementwise_affine_weight)
        nn.init.zeros_(self.elementwise_affine_bias)

    def forward(self, x: Tensor) -> Tensor:
        assert x.ndim == 2  # (batch_size, n_features)
        assert x.shape[-1] == self.weight.shape[0]

        x_input = x
        # x: (B, n_features) -> (B, n_features, d_embedding - int(concat_input))
        x = (
            self.weight * x[..., None]
            if self.bias is None
            else torch.addcmul(self.bias, self.weight, x[..., None])
        )
        x = torch.cos(2 * torch.pi * x)
        if self._concat_input:
            # The input goes first to avoid the potential subsequent masking.
            x = torch.cat([x_input[..., None], x], dim=-1)
        x = torch.addcmul(
            self.elementwise_affine_bias, self.elementwise_affine_weight, x
        )

        return x
```

## EXIT · write before checking the solution

Explain the three mechanisms, design four controlled cells plus tree/RealMLP baselines, state selection and compute budgets, give two falsification tests and one revision condition. Explain the five-versus-ten seed discrepancy and the observer. Revisit in 1, 7 and 30 days. Ask the agent for feedback. Author checks never grade your written defense.

In [10]:
submission={'mechanism_distinction':'','factorial_controls':'','baseline_rationale':'','selection_budget':'','falsification_tests':[],'revision_condition':'','source_discrepancies':'','status':'PENDING_WRITTEN_DEFENSE'}
Path('b02-submission.json').write_text(json.dumps(submission,indent=2)+'\n')

242